In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time

In [2]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [3]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [4]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error'   ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import folder name
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
print(folder)

Health_1
CPS
FOODSEC
County
Counties
Percentages: No
Margin of error: No
Number of variables: 4
2009
2021


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
## For csv files:
# remove state FIPS field
# remove "All" category for race/ethnicity
# make sure index is removed
# make sure proportions are now percentages

if geography == 'Tract':
    df_acs1_csv = df_acs1.drop(['State FIPS', 'County FIPS', 'Tract ID'], axis = 1)
    
    if len(unique(df_acs1.Race_Ethnicity.values)) > 1:
        df_acs1_csv = df_acs1_csv[df_acs1_csv['Race_Ethnicity'] != 'All']
    
    df_acs1_csv = df_acs1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_acs1_csv.columns = [col.lower() for col in df_acs1_csv.columns]
    df_acs1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_acs1_csv.columns]


if geography == 'County':
    df_acs1_csv = df_acs1.drop(['State FIPS', 'County FIPS'], axis = 1)
    df_mpo1_csv = df_mpo1.drop(['State FIPS'               ], axis = 1)
    
    if len(unique(df_acs1.Race_Ethnicity.values)) > 1:
        df_acs1_csv = df_acs1_csv[df_acs1_csv['Race_Ethnicity'] != 'All']
        df_mpo1_csv = df_mpo1_csv[df_mpo1_csv['Race_Ethnicity'] != 'All']
    
    df_acs1_csv = df_acs1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_mpo1_csv = df_mpo1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    
    df_acs1_csv.columns = [col.lower() for col in df_acs1_csv.columns]
    df_mpo1_csv.columns = [col.lower() for col in df_mpo1_csv.columns]
    
    df_acs1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_acs1_csv.columns]
    df_mpo1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_mpo1_csv.columns]

if geography == 'MSA':
    
    if len(unique(df_acs1.Race_Ethnicity.values)) > 1:
        df_acs1_csv = df_acs1[df_acs1['Race_Ethnicity'] != 'All']
    else:
        df_acs1_csv = df_acs1.copy()
        
    df_acs1_csv  = df_acs1_csv.reset_index(drop = True).reset_index().rename(columns = {'index':'python_index'})
    df_acs1_csv.columns = [x.lower() for x in df_acs1_csv.columns]
    df_acs1_csv.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_acs1_csv.columns]


if geography == 'PUMA':
    df_acs.columns = [re.sub('_desc', '', col) for col in df_acs.columns]
    df_acs_csv = df_acs.rename(columns = {'Year':'year', 'state': 'State FIPS'})



In [ ]:
# Set output name for .csv files

if geography == 'Tract':
    name_output_tract_csv  = [indicator_name, '_Tract_' , estimate, '.csv']
    name_output_tract_csv  = "".join(name_output_tract_csv )
    
if geography == 'County':
    name_output_MPO_xlsx = [indicator_name, ' ', 'MPO', ' ', estimate, '.xlsx']
    name_output_MPO_xlsx = "".join(name_output_MPO_xlsx)
    name_output_county_csv = [indicator_name, '_County_', estimate, '.csv']
    name_output_MPO_csv    = [indicator_name, '_MPO_'   , estimate, '.csv']
    name_output_county_csv = "".join(name_output_county_csv)
    name_output_MPO_csv    = "".join(name_output_MPO_csv   )
    
if geography == 'MSA':
    name_output_MSA_csv = [indicator_name, '_MSA_', estimate, '.csv']
    name_output_MSA_csv = "".join(name_output_MSA_csv)

if geography == 'PUMA':
    name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
    name_output_PUMA_csv = "".join(name_output_PUMA_csv)

In [ ]:
# Set file path for exporting
path_out_csv  = os.path.join(path_agol, indicator_name)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tract':
    df_acs1_csv.to_csv(os.path.join(path_out_csv, name_output_tract_csv), index = False)

if geography == 'County':
    df_acs1_csv.to_csv(os.path.join(path_out_csv, name_output_county_csv), index = False)
    df_mpo1_csv.to_csv(os.path.join(path_out_csv, name_output_MPO_csv   ), index = False)

if geography == 'MSA':
    df_acs1_csv.to_csv(os.path.join(path_out_csv, name_output_MSA_csv), index = False)

if geography == 'PUMA':
    df_acs_csv.to_csv(os.path.join(path_out_csv, name_output_PUMA_csv), index = False)
    

print('')
print("Successfully exported")